In [ ]:
import pandas as pd
import json
import re
import random
import textwrap
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageChops
from fontTools.ttLib import TTFont

# =========================
# 1. 폴더, 파일 경로 설정
# =========================
BASE_DIR = Path(".")
WORK_DIR = BASE_DIR / "Ref C (Font)"

FONTS_DIR       = WORK_DIR / "font_ttf_files"
JSON_FRONT_PATH = WORK_DIR / "front_labels.jsonl"
CSV_BACK_PATH   = WORK_DIR / "back_label_200.csv"

OUT_FRONT_BW    = WORK_DIR / "front_bw"
OUT_FRONT_COLOR = WORK_DIR / "front_color"
OUT_BACK_BW     = WORK_DIR / "back_bw"
OUT_BACK_COLOR  = WORK_DIR / "back_color"

for d in [OUT_FRONT_BW, OUT_FRONT_COLOR, OUT_BACK_BW, OUT_BACK_COLOR]:
    d.mkdir(parents=True, exist_ok=True)

FRONT_FONT_SIZE, FRONT_SMALL_RATIO, FRONT_LINE_GAP, FRONT_PAD = 160, 0.6, 20, 20
FRONT_SLASH_POLICY, FRONT_ADD_EMPTY_LINE, FRONT_PIXEL_CROP = "split", True, True
BACK_FONT_SIZE, BACK_LINE_SPACING, BACK_PADDING_X, BACK_PADDING_Y, BACK_MAX_CHARS_PER_LINE = 42, 10, 50, 50, 35
OUTPUT_SIZE, SKIP_MISSING_GLYPHS = 1024, True

VOLUME_PATTERN = re.compile(r"^(.*?)\s*(\d+(?:\.\d+)?\s*(?:ml|mL|g|kg|oz|pads|sheets|ea))\s*$", re.IGNORECASE)
RESAMPLE_LANCZOS = getattr(Image.Resampling, "LANCZOS", Image.LANCZOS)

# =========================
# 2. 데이터 로드
# =========================
back_records, front_records = [], []
try:
    df = pd.read_csv(CSV_BACK_PATH)
    col_name = 'text' if 'text' in df.columns else df.columns[0]
    back_records = df[col_name].dropna().astype(str).tolist()[:200]
except Exception as e:
    print(f"❌ 후면 데이터 로드 실패: {e}")

try:
    with open(JSON_FRONT_PATH, "r", encoding="utf-8") as f:
        front_records = [{"id": f"{i+1:05d}", **json.loads(line.strip())} for i, line in enumerate(f) if line.strip()][:200]
except Exception as e:
    print(f"❌ 전면 데이터 로드 실패: {e}")

# =========================
# 3. 공통 유틸리티
# =========================
def get_clean_font_name(p: Path) -> str:
    return re.sub(r"[^a-zA-Z0-9가-힣]", "", p.stem)

def get_random_dark_color():
    rgb = [random.randint(50, 150) for _ in range(3)]
    rgb[random.randint(0, 2)] = random.randint(0, 50)
    return tuple(rgb)

def best_cmap(font_path: Path) -> dict:
    try:
        with TTFont(str(font_path), lazy=True) as tt:
            return tt["cmap"].getBestCmap() or {}
    except:
        return {}

def missing_chars(cmap: dict, text: str):
    return sorted([ch for ch in set(text) if not ch.isspace() and ord(ch) not in cmap])

def fit_into_square(img: Image.Image, size: int):
    w, h = img.size
    scale = min(size / w, size / h, 1.0)
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    img_resized = img.resize((nw, nh), resample=RESAMPLE_LANCZOS)
    canvas = Image.new("RGB", (size, size), (255, 255, 255))
    ox, oy = (size - nw) // 2, (size - nh) // 2
    canvas.paste(img_resized, (ox, oy))
    return canvas, float(scale), (int(ox), int(oy))

def tight_crop_white(img: Image.Image) -> Image.Image:
    if not FRONT_PIXEL_CROP: return img
    bbox = ImageChops.difference(img, Image.new("RGB", img.size, (255, 255, 255))).getbbox()
    return img.crop(bbox) if bbox else img

# =========================
# 4. 전면/후면 렌더링
# =========================
def render_front_block(lines, font_path: Path, text_color: tuple) -> Image.Image:
    font_main = ImageFont.truetype(str(font_path), FRONT_FONT_SIZE)
    font_small = ImageFont.truetype(str(font_path), int(FRONT_FONT_SIZE * FRONT_SMALL_RATIO))
    dummy_draw = ImageDraw.Draw(Image.new("RGB", (1, 1)))

    parsed, total_h, max_w = [], 0, 0
    for line in lines:
        font = font_small if VOLUME_PATTERN.match(line) else font_main
        w, h = dummy_draw.textbbox((0,0), line, font=font)[2:] if hasattr(dummy_draw, "textbbox") else dummy_draw.textsize(line, font)
        if not line.strip(): h = FRONT_FONT_SIZE
        parsed.append({"text": line, "font": font, "w": w, "h": h})
        max_w, total_h = max(max_w, w), total_h + h

    img_w, img_h = max_w + 2 * FRONT_PAD, total_h + max(0, len(lines)-1) * FRONT_LINE_GAP + 2 * FRONT_PAD
    img, current_y = Image.new("RGB", (img_w, img_h), (255, 255, 255)), FRONT_PAD
    draw = ImageDraw.Draw(img)

    for p in parsed:
        if p["text"].strip():
            draw.text(((img_w - p["w"]) // 2, current_y), p["text"], font=p["font"], fill=text_color)
        current_y += p["h"] + FRONT_LINE_GAP
    return img

def render_back_block(full_text, font_path: Path, text_color: tuple):
    font = ImageFont.truetype(str(font_path), BACK_FONT_SIZE)
    wrapped = [line for para in full_text.split('\n') for line in (textwrap.wrap(para, BACK_MAX_CHARS_PER_LINE) if para.strip() else [""])]
    dummy_draw = ImageDraw.Draw(Image.new("RGB", (1, 1)))

    heights, max_w = [], 0
    for line in wrapped:
        if not line: heights.append(int(BACK_FONT_SIZE * 0.5)); continue
        w, h = dummy_draw.textbbox((0,0), line, font=font)[2:] if hasattr(dummy_draw, "textbbox") else dummy_draw.textsize(line, font)
        heights.append(h); max_w = max(max_w, w)

    img_w, img_h = max_w + 2 * BACK_PADDING_X, sum(heights) + max(0, len(wrapped)-1) * BACK_LINE_SPACING + 2 * BACK_PADDING_Y
    img, current_y = Image.new("RGB", (img_w, img_h), (255, 255, 255)), BACK_PADDING_Y
    draw = ImageDraw.Draw(img)

    for i, line in enumerate(wrapped):
        if line:
            draw.text((BACK_PADDING_X, current_y), line, font=font, fill=text_color)
        current_y += heights[i] + BACK_LINE_SPACING
    return img, wrapped

# =========================
# 5. 메인 프로세스
# =========================
def process_font_all(font_path: Path):
    clean_font_name = get_clean_font_name(font_path) or "UnknownFont"
    paths = {
        "f_bw": OUT_FRONT_BW / clean_font_name, "f_co": OUT_FRONT_COLOR / clean_font_name,
        "b_bw": OUT_BACK_BW / clean_font_name,  "b_co": OUT_BACK_COLOR / clean_font_name
    }

    if all((p / "annotations.jsonl").exists() for p in paths.values()):
        print("⏩ 이미 처리됨 (스킵)", flush=True); return

    cmap = best_cmap(font_path)
    if not cmap:
        print("❌ 폰트 파일 오류 (인식 불가)", flush=True); return

    for p in paths.values(): p.mkdir(parents=True, exist_ok=True)

    saved_front, skipped_front = 0, 0
    saved_back, skipped_back = 0, 0

    # --- 🌟 [A] 전면(Front) 데이터 생성 ---
    if front_records and not ((paths["f_bw"]/"annotations.jsonl").exists() and (paths["f_co"]/"annotations.jsonl").exists()):
        miss_front_logs = []

        with open(paths["f_bw"]/"annotations.jsonl", "w", encoding="utf-8") as af_bw, \
             open(paths["f_co"]/"annotations.jsonl", "w", encoding="utf-8") as af_co:

            for rec in front_records:
                rid, lines = rec["id"], rec.get("lines", [])
                text_join = "\n".join(lines)

                miss = missing_chars(cmap, text_join)
                if miss:
                    miss_front_logs.append(json.dumps({"id": rid, "font": str(font_path), "missing_chars": miss}, ensure_ascii=False))
                    if SKIP_MISSING_GLYPHS:
                        skipped_front += 1
                        continue

                for is_color, f_out, d_out, suffix in [(False, af_bw, paths["f_bw"], ""), (True, af_co, paths["f_co"], "_color")]:
                    try:
                        color = get_random_dark_color() if is_color else (0, 0, 0)
                        img = tight_crop_white(render_front_block(lines, font_path, color))
                        final_img, scale, off = fit_into_square(img, OUTPUT_SIZE)

                        fn = f"{clean_font_name}_front{suffix}_{rid}.png"
                        final_img.save(d_out / fn, "PNG")
                        f_out.write(json.dumps({"id": rid, "file_name": fn, "type": "front_label", "original_text": text_join,
                                                "lines": lines, "text_color_rgb": color, "fit_scale": scale, "paste_offset": off}, ensure_ascii=False) + "\n")
                    except Exception: pass
                saved_front += 1

        if miss_front_logs:
            with open(paths["f_bw"]/"missing_glyphs.jsonl", "w", encoding="utf-8") as mf_f:
                mf_f.write("\n".join(miss_front_logs) + "\n")

    # --- 🌟 [B] 후면(Back) 데이터 생성 ---
    if back_records and not ((paths["b_bw"]/"annotations.jsonl").exists() and (paths["b_co"]/"annotations.jsonl").exists()):
        miss_back_logs = []

        with open(paths["b_bw"]/"annotations.jsonl", "w", encoding="utf-8") as ab_bw, \
             open(paths["b_co"]/"annotations.jsonl", "w", encoding="utf-8") as ab_co:

            for idx, text in enumerate(back_records):
                rid = f"{idx+1:05d}"
                miss = missing_chars(cmap, text)
                if miss:
                    miss_back_logs.append(json.dumps({"id": rid, "font": str(font_path), "missing_chars": miss}, ensure_ascii=False))
                    if SKIP_MISSING_GLYPHS:
                        skipped_back += 1
                        continue

                for is_color, f_out, d_out, suffix in [(False, ab_bw, paths["b_bw"], ""), (True, ab_co, paths["b_co"], "_color")]:
                    try:
                        color = get_random_dark_color() if is_color else (0, 0, 0)
                        img, proc_lines = render_back_block(text, font_path, color)
                        final_img, scale, off = fit_into_square(img, OUTPUT_SIZE)

                        fn = f"{clean_font_name}_back{suffix}_{rid}.png"
                        final_img.save(d_out / fn, "PNG")
                        f_out.write(json.dumps({"id": rid, "file_name": fn, "type": "back_label", "original_text": text,
                                                "processed_lines": proc_lines, "text_color_rgb": color, "fit_scale": scale, "paste_offset": off}, ensure_ascii=False) + "\n")
                    except Exception: pass
                saved_back += 1

        if miss_back_logs:
            with open(paths["b_bw"]/"missing_glyphs.jsonl", "w", encoding="utf-8") as mb_b:
                mb_b.write("\n".join(miss_back_logs) + "\n")

    # 로그 출력
    log_msg = f"완료 ✔️ (전면: 흑백 {saved_front}장/컬러 {saved_front}장, 후면: 흑백 {saved_back}장/컬러 {saved_back}장)"

    if skipped_front > 0 or skipped_back > 0:
        log_msg += f" | ⚠️ 누락스킵 (전면: {skipped_front}건, 후면: {skipped_back}건)"

    print(log_msg, flush=True)

# =========================
# 6. 실행
# =========================
if __name__ == "__main__":
    ttf_paths = sorted([p for p in FONTS_DIR.rglob("*.ttf") if p.is_file()])

    print("\n" + "="*60)
    print(" Font Dataset Generator (Front/Back, B&W/Color)")
    print(f"📁 Working Directory: {WORK_DIR.resolve()}")
    print(f"📁 데이터 로드 완료 -> 전면: {len(front_records)}건, 후면: {len(back_records)}건")
    print(f"📁 대상 폰트: 총 {len(ttf_paths)}개")
    print("="*60 + "\n")

    for i, font_path in enumerate(ttf_paths):
        print(f"👉 [{i+1}/{len(ttf_paths)}] {font_path.name} ... ", end="", flush=True)
        try:
            process_font_all(font_path)
        except KeyboardInterrupt:
            print("\n🛑 사용자에 의해 중단되었습니다.")
            break
        except Exception as e:
            print(f"❌ 에러 발생: {e}", flush=True)

    print("\n✅ 모든 작업이 완료되었습니다.")